In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42

df = (
    pd.read_csv("data.csv")
    .reset_index()
    .rename(columns={"index": "source_row"})
)

chunk_cols = [f"chunk_{i}" for i in range(1, 9)]
label_cols = ["binary_relevancy", "binary_faithfulness"]


# ---------- Validate label types / 验证标签类型 ----------

for col in label_cols:
    if df[col].dtype != bool:
        mapping = {
            True: True,
            False: False,
            "True": True,
            "False": False,
            1: True,
            0: False,
        }
        df[col] = df[col].map(mapping)

    if df[col].isna().any():
        raise ValueError(
            f"Unknown values were found in column '{col}'. "
            f"/ 列“{col}”中发现了未知值。"
        )


# An answer is reliable only if it is both relevant and faithful.
# 只有当回答同时具有相关性和忠实性时，才被视为可靠。
df["binary_reliability"] = (
    df["binary_relevancy"] & df["binary_faithfulness"]
).astype(int)


# Count the number of non-empty retrieved chunks.
# 统计每个样本中非空检索文本块的数量。
df["chunk_count"] = df[chunk_cols].notna().sum(axis=1)


# ============================================================
# Extract the last substantive client request
# 提取最后一条有实际内容的客户请求
# ============================================================

ROLE_RE = re.compile(
    r"\b(?:"
    r"оператор\w*|"
    r"сотрудник\w*|"
    r"специалист\w*|"
    r"менеджер\w*|"
    r"жив\w*\s+человек\w*"
    r")\b",
    re.IGNORECASE,
)

HANDOFF_ACTION_RE = re.compile(
    r"\b(?:"
    r"соедин\w*|"
    r"связ\w*|"
    r"перевед\w*|"
    r"позов\w*|"
    r"вызов\w*|"
    r"нуж\w*|"
    r"помощ\w*|"
    r"хочу"
    r")\b",
    re.IGNORECASE,
)

GREETING_RE = re.compile(
    r"^(?:здравствуйте|добрый день|добрый вечер|привет)[,!\s.-]*",
    re.IGNORECASE,
)


def extract_client_turns(dialog: object) -> list[str]:
    """
    Extract all client messages from a full dialogue.
    从完整对话中提取所有客户消息。
    """

    if not isinstance(dialog, str):
        return []

    turns = []

    for part in dialog.split("Клиент:")[1:]:
        turn = part.split("Ассистент:", 1)[0].strip()

        if turn:
            turns.append(turn)

    return turns


def is_handoff_only(text: str) -> bool:
    """
    Determine whether a short message only requests a transfer
    to a human support agent.

    判断一条较短的消息是否仅用于请求转接人工客服。
    """

    normalized = " ".join(text.lower().split())
    normalized = GREETING_RE.sub("", normalized).strip(" ,.!?")

    if not normalized:
        return False

    # A long and meaningful message should not be treated as
    # a handoff-only request.
    #
    # 较长且包含实际内容的消息不应被视为仅请求转人工。
    if len(normalized.split()) > 8:
        return False

    has_role = bool(ROLE_RE.search(normalized))
    has_action = bool(HANDOFF_ACTION_RE.search(normalized))

    # Examples:
    # "оператор", "оператора", "живой человек".
    #
    # 示例：
    # “оператор”、“оператора”、“живой человек”。
    role_only = has_role and not re.search(
        r"\b(?:"
        r"почему|"
        r"когда|"
        r"сколько|"
        r"комисси\w*|"
        r"сч[её]т\w*|"
        r"карт\w*|"
        r"перевод\w*|"
        r"деньг\w*|"
        r"кредит\w*"
        r")\b",
        normalized,
        re.IGNORECASE,
    )

    return has_role and (has_action or role_only)


def extract_last_substantive_question(dialog: object) -> str | None:
    """
    Return the last client message that contains a substantive request.

    返回最后一条包含实际请求内容的客户消息。
    """

    turns = extract_client_turns(dialog)

    for turn in reversed(turns):
        if not is_handoff_only(turn):
            return turn

    # Important: do not use the last handoff request as a fallback.
    # 重要：不要将最后一条转人工请求作为备用问题返回。
    return None


df["question"] = df["full_dialog"].map(
    extract_last_substantive_question
)

df["has_question"] = (
    df["question"].notna()
    & df["question"].str.strip().ne("")
)

no_question_df = df.loc[~df["has_question"]].copy()

print(
    "Rows without a substantive client question "
    "/ 缺少实质性客户问题的行数:",
    len(no_question_df),
)


# Keep only examples with a substantive question for the main
# relevance and reliability benchmark.
#
# 主相关性与可靠性基准仅保留包含实质性问题的样本。
df = df.loc[df["has_question"]].copy()


# ============================================================
# Exact duplicates and conflicting annotations
# 完全重复样本与冲突标注
# ============================================================

# Use the complete dialogue rather than only the extracted
# final question when identifying duplicate examples.
#
# 在识别重复样本时，使用完整对话，而不仅是提取出的最后一个问题。
key_cols = ["full_dialog", "answer"] + chunk_cols


# When duplicate rows exist, prefer the row containing
# the most detailed error marker.
#
# 如果存在重复行，优先保留错误标记信息最详细的行。
df["_marker_length"] = df["markers"].fillna("").str.len()

df = df.sort_values(
    ["_marker_length", "source_row"],
    ascending=[False, True],
)


# Exact duplicates have identical inputs and identical labels.
# 完全重复样本具有相同的输入和相同的标签。
full_duplicate_mask = df.duplicated(
    subset=key_cols + label_cols,
    keep="first",
)

full_duplicates_df = df.loc[full_duplicate_mask].copy()
df = df.loc[~full_duplicate_mask].copy()

print(
    "Exact duplicate rows removed "
    "/ 删除的完全重复行数:",
    len(full_duplicates_df),
)


# After exact duplicates are removed, repeated input keys indicate
# conflicting binary annotations.
#
# 删除完全重复项后，如果输入键仍然重复，则表示二元标注存在冲突。
conflict_mask = df.duplicated(
    subset=key_cols,
    keep=False,
)

conflicts_df = df.loc[conflict_mask].copy()
df = df.loc[~conflict_mask].copy()

print(
    "Rows with conflicting labels removed "
    "/ 删除的标签冲突行数:",
    len(conflicts_df),
)


df = (
    df.sort_values("source_row")
    .drop(columns="_marker_length")
    .reset_index(drop=True)
)

print(
    "Rows remaining in the main dataset "
    "/ 主数据集中剩余的行数:",
    len(df),
)


# ============================================================
# Grouped and stratified train/validation/test split
# 分组分层的训练集、验证集和测试集划分
# ============================================================

def normalize_dialog(text: object) -> str:
    """
    Normalize a dialogue before creating dialogue groups.

    在创建对话分组之前对对话文本进行标准化。
    """

    if not isinstance(text, str):
        return ""

    text = text.lower().strip()
    return re.sub(r"\s+", " ", text)


df["dialog_group_text"] = df["full_dialog"].map(
    normalize_dialog
)

df["dialog_group_id"] = pd.factorize(
    df["dialog_group_text"]
)[0]


# Stratify by the joint combination of relevancy and faithfulness.
# 按相关性与忠实性标签的联合组合进行分层。
strat = (
    df["binary_relevancy"].astype(int).astype(str)
    + "_"
    + df["binary_faithfulness"].astype(int).astype(str)
)


# Create 20 approximately equal grouped folds:
# 14 folds for training, 3 for validation, and 3 for testing.
#
# 创建20个大小近似相等的分组折：
# 14折用于训练，3折用于验证，3折用于测试。
splitter = StratifiedGroupKFold(
    n_splits=20,
    shuffle=True,
    random_state=SEED,
)

fold_ids = np.full(
    len(df),
    -1,
    dtype=int,
)

for fold_id, (_, fold_indices) in enumerate(
    splitter.split(
        X=df,
        y=strat,
        groups=df["dialog_group_id"],
    )
):
    fold_ids[fold_indices] = fold_id


if (fold_ids < 0).any():
    raise RuntimeError(
        "Some rows were not assigned to a fold. "
        "/ 部分行未被分配到任何折中。"
    )


df["fold"] = fold_ids

df["split"] = np.select(
    [
        df["fold"] < 14,
        df["fold"] < 17,
    ],
    [
        "train",
        "val",
    ],
    default="test",
)


# ============================================================
# Verify that no dialogue group leaks across dataset splits
# 验证不同数据集划分之间不存在对话组泄漏
# ============================================================

train_groups = set(
    df.loc[
        df["split"] == "train",
        "dialog_group_id",
    ]
)

val_groups = set(
    df.loc[
        df["split"] == "val",
        "dialog_group_id",
    ]
)

test_groups = set(
    df.loc[
        df["split"] == "test",
        "dialog_group_id",
    ]
)

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)


# ============================================================
# Split diagnostics
# 数据集划分诊断
# ============================================================

for split_name in ["train", "val", "test"]:
    part = df.loc[df["split"] == split_name]

    print(
        f"\nSplit: {split_name} "
        f"/ 数据集划分: {split_name}"
    )

    print(
        "Number of rows / 行数:",
        len(part),
    )

    print(
        "\nJoint label distribution "
        "/ 联合标签分布:"
    )

    print(
        pd.crosstab(
            part["binary_relevancy"],
            part["binary_faithfulness"],
        )
    )

    print(
        "\nReliable class proportion "
        "/ 可靠类别占比:",
        part["binary_reliability"].mean(),
    )

    print(
        "Chunk-count distribution "
        "/ 文本块数量分布:",
        (
            part["chunk_count"]
            .value_counts(normalize=True)
            .sort_index()
            .to_dict()
        ),
    )


# ============================================================
# Save the processed datasets
# 保存处理后的数据集
# ============================================================

df.loc[df["split"] == "train"].to_csv(
    "train.csv",
    index=False,
)

df.loc[df["split"] == "val"].to_csv(
    "val.csv",
    index=False,
)

df.loc[df["split"] == "test"].to_csv(
    "test.csv",
    index=False,
)

no_question_df.to_csv(
    "quarantine_no_question.csv",
    index=False,
)

full_duplicates_df.to_csv(
    "removed_full_duplicates.csv",
    index=False,
)

conflicts_df.to_csv(
    "quarantine_label_conflicts.csv",
    index=False,
)


print(
    "\nProcessing completed successfully. "
    "/ 数据处理已成功完成。"
)